<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_G07_MULTISCALE_FAULT_DISCOVERY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS G07 - high-resolution multiscale fault discovery (code rev g07.1-r2)

Four coordinated systems: **A** surface (USGS 3DEP 1 m DEM geomorphology), **B** subsurface (REUSED G06 feature store, improved supervision), **C** candidate hypotheses + learned relevance, **D** DTI-aware exact-budget selection and gated export.

**This notebook's first run goes end to end on a GPU runtime:** Drive -> DEM validation -> feasibility -> DEM features -> datasets -> **U-Net on CUDA** -> classical models -> exact-budget evaluation -> reports. Run it top to bottom on a GPU runtime (Runtime -> Change runtime type -> GPU). Mandatory stages **raise** with the reason instead of letting later cells print results that do not exist (`STRICT_NOTEBOOK`).

| where it runs | sections |
|---|---|
| **GPU** | 14 (U-Net training and its patch inference) |
| CPU | everything else: DEM processing, datasets, LightGBM / Random Forest (their registry `device` says `cpu`), evaluation |

* Outputs: `MyDrive/GEMS_2/outputs/g07_multiscale_fault_discovery/<run_id>/` (run id = data-engineering identity; replaced outputs go to `history/`).
* Rerunning after a disconnect resumes: DEM tiles, models and epoch checkpoints are persisted as they finish. The G06 store is never rebuilt.
* **Nothing is uploaded.** The 0.40 leaderboard DTI is a target, never inferred from offline proxies. Docs: `docs/g07/`.

## 00 - Research objective and historical findings

**Hypothesis:** high-resolution surface morphology, improved fault supervision and independently generated subsurface hypotheses reveal structures W2 misses; a learned ranking + DTI-aware selection converts them into better predictions.

**Historical lessons (G04-G06), enforced in code:** W2 is the strongest reference; naive magnetic / gravity fusion lowered DTI; segment means destroy W2's pixel ranking (G07 keeps an exact-k W2 base); conservative corrections became inactive; G06 segment models had too few positives (G07 adds geometric positive augmentation of TRAINING truth); G06 exact-area evaluations were corrupted by ties (G07 selects exactly k pixels with label-free tie-breaking); geology-only models ranked withheld faults worse than W2 (G07 tests label design before model size).

| id | question |
|---|---|
| A0 | W2 exact-k reference: reproduce the baseline |
| A1 / A2 | does the DEM carry independent fault evidence / do scarps and terrain continuity help? |
| B1 / B2 | was label design limiting geology-only ML? / does a distance target improve localisation? |
| C1 | does spatial DEM context (U-Net) help? |
| C2 | do surface + subsurface candidate unions recover more systems at matched budget? |
| D1 / D2 | can a ranker separate true structures from false edges? / W2-preserving residual selection |
| E1 | continuous confidence vs binary top-area |

References (other protocols, never comparable): W2 V1 0.282538, W2 V3 0.130868, public 0.0942.

**DTI-first research update (code branch):** this notebook now includes section 09A for DEM/W2 paired ΔDTI, and 16A for cached model DTI without retraining. These reports are descriptive; original V1/V3 validation and strict promotion remain authoritative. Existing AUC charts are legacy diagnostics, not selection objectives.


## 01 - CONFIG and RUN controls

The defaults below are the first GPU run. **Training mode:** `dem.require_go=True` is `STRICT_RESEARCH` (DEM-based training needs a feasibility GO). `dem.require_go=False` is `EXPLORATORY_TRAINING`: it also trains after NO_GO / INCONCLUSIVE, but every DEM-based result is `SCREENING_ONLY` and can never be promoted or exported; missing files, invalid CRS, unknown vertical units or leakage are never bypassed. **`dem.vertical_unit_override` stays unset** until section 04 shows it is needed and you have checked the USGS metadata. Later runs: `docs/g07/COLAB_EXECUTION.md`.

In [ ]:
CONFIG = {
    "repo": "mybenkhadda/GEMS---drivendata",
    "ref": "g07-dti-first-metrics",  # DTI code branch; after PR merge use main                         # pin the pushed commit SHA for a reproducible run
    "drive_root": "/content/drive/MyDrive/GEMS_2",
    "local_root": "/content/gems_work",
    "output_subdir": "outputs/g07_multiscale_fault_discovery",
    "seed": 2026,

    "dem_resolution_target_m": 1.0,        # validated against the real DEM headers (never assumed)
    "dem_tile_px": 512,
    "dem_halo_px": 128,                    # minimum; the operator support raises it when needed
    "analysis_resolution_m": 100.0,

    "use_gpu": True,                       # the U-Net refuses to fall back to CPU
    "reuse_g06_features": True,
    "reuse_w2_predictions": True,

    "g06_run_dir": (
        "/content/drive/MyDrive/GEMS_2/outputs/"
        "g06_geological_feature_store/g06-cda34ef1f1"
    ),

    "strict_provenance": True,
    "save_figures": True,
    "reuse_cached": True,
    "sync_to_drive": True,
    "run_synthetic_tests": True,
    "auto_submit": False,                  # cannot be enabled
}
RUN = {
    "PREFLIGHT": True,
    "DEM_INVENTORY": True,
    "DEM_COVERAGE": True,
    "DEM_FEASIBILITY": True,
    "DEM_FEATURES": True,

    "BUILD_DATASETS": True,

    "TRAIN_SURFACE_CLASSIC": True,
    "TRAIN_SUBSURFACE": True,
    "TRAIN_SURFACE_UNET": True,

    "GENERATE_CANDIDATES": False,
    "TRAIN_CANDIDATE_RANKER": False,

    "EVALUATE": True,

    "FULL_INFERENCE": False,
    "EXPORT_SUBMISSION": False,

    "REPORT": True,
}
OVERRIDES = {                              # deep-merged into configs/g07/discovery.yaml
    "dem": {
        "download": {"enabled": False},
        "require_go": False,               # DEM pilot is INCONCLUSIVE; SCREENING_ONLY
        "work_res_m": 2.0,
        # "vertical_unit_override": "metre",   # only after section 04 asks for it and the metadata is checked
    },
    "surface": {
        "unet": {
            "epochs": 20,
            "batch": 4,
            "steps_per_epoch": 40,
            "patience": 5,
            "base_channels": 16,
            "depth": 3,
        }
    }
}
STRICT_NOTEBOOK = True   # True: a mandatory stage that did not succeed raises; False: print it and continue (batch use)
PRIVATE_REPO = False
EXPECT = {"code_rev": "g07.1-r2", "commit": None}
# Fixed, predeclared DTI evaluation settings. Never tune these on V1/V3 held-out truth.
DTI_DIAGNOSTICS = {
    "enabled": True,
    "cached_models": True,
    "fracs": [0.005, 0.01, 0.02],
    "fusion_weights": [0.15, 0.35],
    "feature_directions": {
        "dem_scarp_s10": "high",
        "dem_scarp_s150": "high",
        "dem_lin_s10": "high",
        "dem_hs_range": "high",
        "dem_dircoh_fine": "high",
    },
}


## 02 - Mount Drive and initialize repository

Restart the session before re-running if `gems_bench` is already imported.

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'g07_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: Runtime -> Restart session, then run from the top')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab,m9]', 'pytest', 'scikit-image'], check=True)
sys.path.insert(0, CODE)
os.chdir(CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('HEAD', HEAD)

In [ ]:
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
import gems_bench.g07 as G07
from gems_bench.g07 import config as GC
from gems_bench.g07 import persistence as PS
from gems_bench.g07.pipeline import G07Pipeline, StageError
assert G07.G07_CODE_REV == EXPECT['code_rev'], G07.G07_CODE_REV
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit'])
cfg = GC.load_config(overrides={'notebook': CONFIG, 'run': RUN, **OVERRIDES})
P = G07Pipeline(cfg, RUN, HEAD)
OUT = P.root

def show(rel, cols=None, n=25):
    p = OUT / rel
    if not p.exists():
        print('ABSENT:', rel); return None
    df = pd.read_csv(p) if p.suffix == '.csv' else pd.read_parquet(p)
    display(df[[c for c in (cols or df.columns) if c in df]].head(n)); return df

def js(rel, n=2000):
    d = P.ws.read_json(rel)
    print(json.dumps(d, indent=1, default=str)[:n] if d is not None else f'ABSENT: {rel}')

def figs(sub, limit=6):
    for f in sorted((OUT / 'figures' / sub).glob('*.png'))[:limit]:
        print(f.name); display(Image(str(f)))

def require(stage, allow=()):
    """Mandatory stage: True when it reached a valid state; otherwise raises (STRICT_NOTEBOOK) or prints why."""
    try:
        P.require_stage(stage, allow)
        return True
    except StageError as e:
        if STRICT_NOTEBOOK:
            raise
        print(f'[not available] {e}')
        return False

print('run id', P.run_id, '| mode', GC.training_mode(cfg), '| local', OUT, '| Drive', P.ws.drive_path())

## 03 - Validate repository commit, existing artifacts and GPU readiness

**Input:** Drive (GeoDAWN raster, faults, template, P6 W2 maps, G06 store). **Operation:** synthetic tests (Colab), frozen-engine attach, artifact checks, **CUDA check**. **Output:** `dependencies/existing_artifacts.csv`, `missing_dependencies.json`, `audit/gpu_runtime.json`. **Purpose:** fail in the first minutes, before any compute is spent: `use_gpu=True` without CUDA raises here.

In [ ]:
P.run_tests(); show('audit/tests.csv')
if CONFIG['run_synthetic_tests']:
    require('tests')
P.preflight()
show('dependencies/existing_artifacts.csv', ['artifact', 'mandatory', 'status', 'detail', 'blocks'], n=20)
js('dependencies/missing_dependencies.json')
require('preflight')
P.gpu_preflight()
js('audit/gpu_runtime.json')
if RUN['TRAIN_SURFACE_UNET']:
    require('gpu_preflight')

## 04 - DEM tile validation and source contract

Existing tiles under `MyDrive/GEMS_2/data/raw/dem_1m/` are classified file by file: **LOCAL** (readable, projected CRS in metres, north-up transform, accepted resolution, explicit vertical unit, intersects the study area), **LOCAL_INVALID**, **LOCAL_OUTSIDE_STUDY_AREA** or **CATALOG_ONLY** (only in `tnm_urls.json`, never treated as data). Nothing is downloaded unless you enable it.

**If all tiles are excluded with `UNKNOWN_VERTICAL_UNIT`:** the headers state no elevation unit. Check one tile's USGS metadata, then set `OVERRIDES['dem']['vertical_unit_override'] = 'metre'` in section 01 and rerun. That is a recorded assumption; the unit is never inferred from the horizontal CRS. A 100 m elevation band is **never** a high-resolution DEM. Licence: USGS 3DEP is public domain; competition eligibility of external data must be confirmed by you (`dem.competition_eligibility_verified`).

In [ ]:
d = P.ws.read_json('manifests/dependencies.json') or {}
print('grid', d.get('grid'))
P.dem_inventory()
require('dem_inventory')
js('dem_catalog/dem_source_validation.json')
T = show('dem_catalog/dem_inventory.csv', ['tile_id', 'status', 'res_m', 'crs', 'vertical_unit', 'vertical_unit_source', 'nodata', 'project', 'year', 'exclusion_codes'], n=15)
print('Excluded local files (every one with its reason):')
show('dem_catalog/dem_invalid_sources.csv', ['tile_id', 'status', 'exclusion_codes', 'problems'], n=30)
show('dem_catalog/source_licenses.csv')

## 05 - DEM catalog and geographic coverage

Coverage is measured on the **eligible fault-discovery region** (footprint & template & not a known fault) and on each V1 evaluation zone - not as bounding-box overlap. Output: `tables/dem_coverage.csv`, `dem_catalog/dem_coverage.geojson`, `figures/dem_atlas/dem_coverage.png`.

In [ ]:
P.dem_coverage()
require('dem_coverage')
show('tables/dem_coverage.csv')
figs('dem_atlas')

## 06 - Full raw-data and DEM atlas

Native-resolution DEM panels (hillshades by azimuth, slope, curvature, roughness, lineaments, scarps) and the 1 m vs 100 m comparison are produced inside the feasibility atlas (section 07) for the representative cases; the raw 19-band atlas is the G06 one (`figures/raw_atlas` of the G06 run).

In [ ]:
T = show('dem_catalog/dem_inventory.csv', n=0)
if T is not None and len(T):
    display(T.groupby(['status', 'project']).size().rename('tiles').reset_index())

## 07 - DEM feasibility status and training gate

Pilot DEM features on the sources covering the most eligible V1-zone area (label-free choice; fine tier stops starting tiles at `feasibility.pilot_budget_hours`), fault signal (AUC fault-near vs background, coverage-artefact screen on |rho|, scale stability, orientation consistency, W2 false-positive support), DEM-score recall vs W2 at the same budget, 20-30 case figures. **Held-out truth is used for this diagnostic only.** Predeclared rule (`feasibility.go_if`): GO / NO_GO / INCONCLUSIVE / BLOCKED_INPUT; **V1 gates, V3 is reported separately**.

**The cell below shows the actual decision and what it allows** before any DEM-based training. In `STRICT_RESEARCH` a NO_GO stops at section 08 with the corrective action; the thresholds are never lowered to force a GO.

**Metric note:** the legacy feasibility GO/NO_GO rule uses AUC as one preliminary evidence gate. The DTI-first assessment is in section **09A**, after the DEM feature raster exists. All held-out feature-DTI results remain diagnostic and cannot select fusion weights or authorize promotion.


In [ ]:
P.assess_dem_feasibility()
require('dem_feasibility', allow=(PS.NO_GO, PS.BLOCKED_INPUT))    # every decision is a result; the gate below acts on it
js('dem_catalog/feasibility_decision.json')
p = OUT / 'reports/DEM_FEASIBILITY.md'
if p.exists(): display(Markdown(p.read_text()))
figs('dem_feasibility', 8)
gate = P.dem_gate()
print('\nTRAINING GATE')
for k in ('mode', 'decision', 'status', 'reason', 'corrective_action', 'gpu_training_can_proceed', 'result_tier', 'n_usable_sources'):
    print(f'  {k:<26} {gate[k]}')
print('  input requirements:', *gate['input_requirements'], sep='\n    - ')

## 08 - Native-resolution DEM feature extraction (CPU)

Allowed by the training gate. First the merged 10 m DEM (all the U-Net needs), then the fine tier at the work resolution (per tile, halo read across tile edges from aligned neighbours, nodata-aware; features are NaN near real data edges, so tile boundaries invent no lineaments), then the coarse tier; exact area-weighted aggregation to 100 m. Every tile product is pushed to Drive when finished. Past `dem.fine_budget_hours` no new tile starts and the stage is **PARTIAL**: training continues on what exists and a rerun resumes. Output: `dem_features/dem_features_100m.tif`, registry, agreement raster, `tables/dem_seam_audit.csv`.

In [ ]:
P.prepare_dem_features()
require('dem_features', allow=(PS.PARTIAL,))
js('dem_features/dem_features_manifest.json')
show('dem_features/dem_feature_registry.csv', n=20)
A = show('tables/dem_seam_audit.csv', n=0)
if A is not None and len(A):
    print(f'{int(A.flag.sum())}/{len(A)} features flagged at DEM tile boundaries')
    display(A[A.flag].sort_values('ratio', ascending=False).head(15))

## 09 - Multiscale terrain structure maps

Scarps, lineaments, truncations, drainage alignment and persistence are observations, not interpretations (a scarp is not automatically tectonic).

In [ ]:
st = P.dem_store()
if st is not None:
    from gems_bench.g07 import visualization as V
    f = 8
    names = [n for n in ('dem_scarp_s10', 'dem_lin_s10', 'dem_scarp_persist_fine', 'dem_dircoh_fine', 'dem_scarp_s150', 'dem_valleyline_fine') if n in st.names]
    from gems_bench.g06 import visualization as V6
    V6.plot_feature_atlas({n: st.load([n])[0][::f, ::f] for n in names}, OUT / 'figures' / 'structural_features', 'terrain_structure', 'Terrain structure (100 m aggregates)', None, 1, 3)
    figs('structural_features')
else:
    print('no DEM feature store (DEM_FEATURES not run / not GO)')

## 09A — DTI-first fault discovery diagnostic (NO TRAINING)

**Primary research metric: matched ΔDTI against W2**, not ROC-AUC. The existing competition-style scorer supplies a 300 m linear support kernel with α=0.2, β=0.8, and the correct V1/V3 exclusion masks. All predictions use predeclared label-free feature directions and exact-k fractions.

**Two noninterchangeable protocols:**

1. **DEM-covered local diagnostic:** DEM and W2 use the *same finite DEM-eligible domain* and the same k. Truth/support/FP are restricted to that domain. These values are *not comparable* to global/leaderboard DTI and can be edge-censored.
2. **Full-region incremental diagnostic:** W2 remains the baseline across the original V1/V3 unit. A small predeclared blend changes only scores inside valid DEM coverage. Both predictions are reselected to the same full-region exact-k area budget before canonical DTI scoring.

Output: `evaluation/dti_screening/{coverage,feature_local_rows,feature_local_summary,fusion_full_rows,fusion_full_summary,system_recovery,manifest}.csv/json` plus PNG/SVG figures. Every output is **SCREENING_ONLY**. Incomplete units or fewer than five independent DEM-covered fault systems do not justify model promotion. This section performs no training and never reads held-out labels to set directions, budgets or weights.


In [ ]:
from gems_bench.g07.dti_screening import run_feature_dti, save_dti_figures

DTI_RESULT = None
if DTI_DIAGNOSTICS["enabled"]:
    if P.dem_store() is None:
        print("DTI feature diagnostic unavailable: DEM features have not been generated.")
    else:
        DTI_RESULT = run_feature_dti(
            P,
            features=DTI_DIAGNOSTICS["feature_directions"],
            fracs=tuple(DTI_DIAGNOSTICS["fracs"]),
            fusion_weights=tuple(DTI_DIAGNOSTICS["fusion_weights"]),
        )
        print("DEM-covered support and independent fault systems:")
        display(DTI_RESULT["coverage"])
        print("LOCAL DEM-only feature ΔDTI vs SAME-COVERAGE W2 (NOT leaderboard DTI):")
        if len(DTI_RESULT["local"]):
            display(DTI_RESULT["local"].query("frac == 0.01 and model != 'W2'")
                    .sort_values(["scenario", "delta_dti"], ascending=[True, False]).head(20))
        print("FULL-UNIT W2+DEM ΔDTI, exact 1% budget (SCREENING_ONLY):")
        if len(DTI_RESULT["full"]):
            display(DTI_RESULT["full"].query("frac == 0.01")
                    .sort_values(["scenario", "delta_dti"], ascending=[True, False]).head(30))
        if len(DTI_RESULT["system_recovery"]):
            display(DTI_RESULT["system_recovery"].head(20))
        saved = save_dti_figures(DTI_RESULT, OUT / "figures" / "dti_screening")
        if saved:
            P.push(["figures/dti_screening"])
            figs("dti_screening", 4)
        js("evaluation/dti_screening/manifest.json", 4000)
        print("Research warning: held-out truth is used only to score predeclared policies; do not select a winner here.")


## 10 - Existing G06 feature-store restoration

The G06 store is **reused** (staged from Drive, validated: files, alignment, schema hash) - never rebuilt.

In [ ]:
P.restore_g06()
require('g06_restore')
js('dependencies/g06_store.json')

## 11 - Complete training visibility and fold audit (read after section 12)

V1 (withheld systems), V3 (geographic), V2 (completion); nested fits `j|k`; explicit isolation tests. These files are written by section 12, so a fresh run prints ABSENT here and the audit is shown again there.

In [ ]:
show('splits/fits.csv', ['fit_id', 'visible_px', 'hidden_px', 'region_px', 'predicts'], n=30)
A = show('splits/isolation_audit.csv', n=0)
if A is not None and len(A): display(A.status.value_counts())

## 12 - Datasets, surface patches and patch readiness

Subsurface pool and label variants, surface pool on DEM-covered cells, analysis-aligned surface patches (10 m). `datasets/surface/patch_readiness.csv` has one row per U-Net fit: training / validation / positive / unlabeled patches, DEM coverage, channels, resolution, READY or the exact reason why not. Only training-visible faults are used. Nested fits are skipped until candidates are requested. A DEM rebuild does not redraw the subsurface pool.

In [ ]:
P.prepare_training_data()
require('datasets')
show('datasets/dataset_summary.csv', n=40)
js('datasets/surface/patches/surface_manifest.json', 800)
A = show('splits/isolation_audit.csv', n=0)
if A is not None and len(A): display(A.status.value_counts())
print('Patch readiness per U-Net fit:')
show('datasets/surface/patch_readiness.csv')

## 13 - Improved subsurface supervision datasets

Pool drawn with the G06 sampler (denser positives); per fit: G06 weights, IMPROVED weights (system-balanced positives, distance-graded PU, stronger hard negatives, full region balance) and the proximity target exp(-d / 0.3 km).

In [ ]:
L = show('datasets/subsurface/labels/V1_0.parquet', n=0)
if L is not None and len(L):
    display(L.groupby('cls').agg(n=('row_id', 'size'), w_g06=('weight_g06', 'sum'), w_improved=('weight_improved', 'sum'), proximity=('proximity', 'mean')))

## 14 - Compact surface U-Net on CUDA (G07-C1) - GPU

Runs **before** the classical benchmarks so CPU fits cannot use up the GPU session. Before training, the stage checks the DEM raster, coverage, coarse DEM, patch manifest and memmap, channel order, resolution and per-fit readiness, and blocks with the exact reason if any fails. A finished fit is reused; an interrupted one resumes from `models/surface_unet/<fp>/<fit>.checkpoint.pt`. On CUDA out-of-memory the registry row says how to recover (lower `batch`, then `base_channels`).

In [ ]:
P.train_surface_unet_stage()
require('train_surface_unet')
print('Per-fit registry (status, device, best epoch):')
show('models/surface_unet_registry.csv', ['experiment', 'fit_id', 'status', 'device', 'strictness', 'best_epoch', 'best_inner_val_loss', 'n_train', 'n_val', 'seconds', 'detail'])
show('models/surface_unet_validation.csv', ['experiment', 'fit_id', 'valid', 'n_predictions_valid', 'problem'])
print('Training history (last epochs) and loss curves:')
H = show('models/surface_unet/training_history.csv', n=0)
if H is not None and len(H):
    display(H.groupby('fit_id').tail(2)[['fit_id', 'epoch', 'train_loss', 'inner_val_loss', 'lr', 'gpu_allocated_mb', 'gpu_reserved_mb', 'seconds', 'best_epoch']])
figs('training', 3)

## 15 - Subsurface improved-supervision benchmarks (G07-B0 / B1 / B2) - CPU LightGBM

Same pool, visibilities and G06-P4 families; only the label design changes. A partial result is saved and marked PARTIAL; a rerun trains only the missing fits. This is CPU work: if the GPU session is short, set `TRAIN_SUBSURFACE` off and run it on a CPU runtime later.

In [ ]:
P.benchmark_subsurface_models()
show('models/subsurface_experiment_status.csv')
show('models/subsurface_registry.csv', ['experiment', 'fit_id', 'status', 'device', 'strictness', 'target', 'n_train', 'best_iteration', 'seconds', 'detail'], n=40)
if RUN['TRAIN_SUBSURFACE']:
    require('train_subsurface', allow=(PS.PARTIAL,))

## 16 - Surface classical ML benchmarks (G07-A1 / A2 / A1RF) - CPU

LightGBM / Random Forest on aggregated DEM channels, DEM-covered cells only. Strictness in the registry follows the training gate (`SCREENING_ONLY` in exploratory mode). LightGBM runs on CPU on purpose: a GPU LightGBM build is not assumed in Colab.

In [ ]:
P.benchmark_surface_models()
show('models/surface_experiment_status.csv')
show('models/surface_registry.csv', ['experiment', 'fit_id', 'status', 'device', 'strictness', 'kind', 'n_train', 'best_iteration', 'seconds', 'detail'], n=40)
if RUN['TRAIN_SURFACE_CLASSIC']:
    require('train_surface_classic', allow=(PS.PARTIAL,))

## 16A — Cached W2/B0/B1/A-series DTI comparison (NO RETRAINING)

Run the competition-style exact-k scorer on any existing held-out model maps. A scenario is pooled **only if every required validation unit has a valid map**; incomplete models remain in per-unit tables. This stage does **not** depend on the model-promotion stage or its historical `v1_gain` KeyError.

Persist: `evaluation/dti_screening/cached_model_{status,unit_rows,complete_pooled,paired_vs_w2}.csv`.


In [ ]:
from gems_bench.g07.dti_screening import cached_models_dti

if DTI_DIAGNOSTICS["enabled"] and DTI_DIAGNOSTICS["cached_models"]:
    DTI_MODELS = cached_models_dti(P, fracs=(0.01,))
    print("Model/scenario fold availability and strictness:")
    display(DTI_MODELS["status"])
    print("Complete-fold pooled DTI at EXACT 1% (no missing folds):")
    if len(DTI_MODELS["pooled"]):
        display(DTI_MODELS["pooled"].sort_values(["scenario", "dti"], ascending=[True, False]))
    print("Paired comparison with W2 (full scenarios only):")
    if len(DTI_MODELS["paired"]):
        display(DTI_MODELS["paired"].sort_values(["scenario", "delta"], ascending=[True, False]))
    else:
        print("No complete model-vs-W2 comparisons; inspect cached_model_status.csv.")
    print("Incomplete-fold unit DTI is stored separately. These are NOT pooled or promoted.")


## 17 - Full candidate inventory (label-free pools; off in the first run)

W2, CNN, surface, subsurface, magnetic, gravity, structural, persistence pools (exact-k), merged and de-duplicated with provenance; candidate recall before ranking (G07-C2). Enabling it also trains the nested inner fits the strict ranker needs.

In [ ]:
P.generate_candidates()
show('candidates/candidate_summary.csv', ['unit', 'n_candidates', 'pools'])
R = show('evaluation/recall/candidate_recall.csv', n=0)
if R is not None and len(R):
    display(R.groupby(['scenario', 'source']).agg(systems=('n_systems', 'sum'), covered=('n_covered', 'sum'), area=('area_frac', 'mean')))

## 18 - Candidate feature engineering

Geometry, G06 geology + DEM means and steps, orientation agreement, permitted known context (nested for training units), honest model evidence, expert agreement, uncertainty. Synthetic candidates are generated **per fit**: for held-out unit `s:z`, from the other units' truth that fit `s:z` may see (`TrainingVisibility.visible`), with their scenario, outer / inner fold, visibility id, method, training systems and upstream provenance on every row. Overlap with held-out geometry raises `LeakageError`.

In [ ]:
D = P._load_candidates()
if len(D):
    display(D.groupby(['unit', 'synthetic', 'outer_fold']).agg(n=('label', 'size'), relevant=('target', 'sum')))
    from gems_bench.g07.models_candidates import column_family
    display(pd.Series([column_family(c) for c in D.columns]).value_counts())
else:
    print('no candidates: GENERATE_CANDIDATES is off in the first run')

## 19 - Geological candidate classification and ranking (G07-D1)

Cross-fitted over units; isotonic calibration from other units' held-out scores; utility; portfolio (D2) and corridor (E1) maps.

In [ ]:
P.train_candidate_ranker()
js('models/candidate_registry.json', 2500)

## 20 - Expert complementarity analysis

Systems each expert recovers that W2 does not, at the same exact budget (filled by section 22).

In [ ]:
show('evaluation/metrics/complementarity.csv', n=30)

## 21 - Exact-budget validation and evaluation status

Every top-area selection has exactly k eligible pixels (deterministic label-free ties); the audit shows what the legacy `>=` rule would have selected. `evaluation_status.csv` says per model and scenario: **FULLY / PARTIALLY / NOT_EVALUATED** and **STRICTLY_VALIDATED / SCREENING_ONLY / BLOCKED**. Only fully evaluated pairs get a pooled DTI; a model missing a fold keeps its per-unit rows in `partial_unit_rows.csv`.

In [ ]:
P.evaluate()
EVALUATED = require('evaluate') if RUN['EVALUATE'] else False
show('evaluation/metrics/evaluation_status.csv', n=40)
show('evaluation/metrics/exact_budget.csv')
figs('model_comparison', 3)

## 22 - V1 complete-system discovery evaluation (primary)

In [ ]:
f = cfg['representation']['primary_frac']
S = None
if EVALUATED:
    S = show('evaluation/metrics/metrics.csv', n=0)
    display(S[(S.scenario == 'V1') & (S.frac == f)][['model', 'rep', 'strictness', 'dti', 'worst_zone_dti', 'tp_w', 'fp_w', 'fn_w', 'area_frac_eligible']].sort_values('dti', ascending=False))
    show('evaluation/metrics/model_comparison.csv', n=30)
else:
    print('EVALUATE did not complete: no DTI to show (see the status table in section 30)')

## 23 - V3 geographic evaluation

In [ ]:
if EVALUATED:
    show('evaluation/geographic/v3_summary.csv', ['model', 'rep', 'frac', 'dti', 'worst_zone_dti', 'zone_dtis'], n=30)
    figs('dti_frontiers', 3)

## 24 - V2 completion diagnostics

In [ ]:
if EVALUATED and S is not None and len(S):
    display(S[S.scenario == 'V2'][['model', 'rep', 'frac', 'dti', 'worst_zone_dti']].sort_values(['frac', 'dti']))

## 25 - Recovered / lost fault-system atlas

Truth outlines are post-hoc only.

In [ ]:
if EVALUATED:
    show('evaluation/metrics/system_recovery.csv', ['model', 'unit', 'rep', 'frac', 'n_systems', 'n_recovered', 'n_recovered_w2', 'n_new', 'n_lost', 'new_ids', 'lost_ids'], n=40)
    figs('recovery_atlas')

## 26 - Feature / model ablations and model promotion eligibility

A model is promoted only with exact budgets, V1 gain, zone guard, V3 guard, net systems and strict validation. A `SCREENING_ONLY` model stops at `PROMISING_SCREENING_ONLY` however good its numbers. The U-Net has V1 fits by default, so its missing V3 evaluation keeps it at `RETAIN_W2`. W2 is retained otherwise.

In [ ]:
if EVALUATED:
    show('evaluation/ablations/ablations.csv', n=30)
    show('evaluation/promotion/promotion_decisions.csv', ['model', 'strictness', 'status', 'v1_gain', 'v3_delta', 'net_systems', 'reasons'], n=20)

## 27 - Full-footprint prediction reconstruction

W2 base = frozen P6 full-region score; the promoted policy (or the W2 exact-k reference) on the competition budget domain; parity checks.

In [ ]:
P.full_inference()
js('predictions/full_region/full_region_manifest.json', 2500)

## 28 - Export and submission parity checks

Gates: EXPORT_SUBMISSION, auto_submit disabled, promoted strict model, full-region parity, DEM competition eligibility (when the DEM is used), provenance. At best READY_FOR_MANUAL_UPLOAD; never uploaded. W2 is retained when anything fails.

In [ ]:
P.export_submission()
js('evaluation/promotion/submission_eligibility.json')

## 29 - Comprehensive scientific report

In [ ]:
P.build_report()
for r in ('DEM_FEASIBILITY', 'DTI_EVALUATION', 'COMPLEMENTARITY', 'SUBMISSION_ELIGIBILITY', 'NEXT_EXPERIMENT'):
    p = OUT / 'reports' / f'{r}.md'
    if p.exists(): display(Markdown(p.read_text()))

## 30 - Execution summary, artifact summary and resume guide

The table is this session's real status; `NOT_REQUESTED` and `NOT_RUN` are not successes. Resume: re-run from the top with the same CONFIG; validated stages are REUSED, DEM tiles, patches, models and U-Net checkpoints are cached by fingerprint. `P.run()` runs everything with the same RUN gates, without raising.

In [ ]:
P.execution_summary()
display(P.stage_table())
inv = show('manifests/artifact_inventory.csv', n=0)
if inv is not None and len(inv):
    display(inv.groupby('section').agg(files=('path', 'size'), mb=('bytes', lambda b: round(b.sum() / 1e6, 1))))
js('manifests/auxiliary_errors.json', 1500)
P.final_message()
# P.run()